# 05 - Visualização Pix

Este notebook cria gráficos formais a partir das tabelas Gold agregadas.

## Regras de Visualização

Os datasets Gold são pequenos e agregados, por isso podem ser convertidos para pandas para geração dos gráficos com matplotlib. Todos os gráficos possuem título, eixos, legenda quando aplicável, grid, fonte de dados e destaque do maior valor.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name in {"notebooks", "i_notebooks"} else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR, PIX_FEE_SAVINGS_DIR, PIX_MONTHLY_INDICATORS_DIR, PIX_TRANSFER_SAVINGS_DIR, create_project_directories
from src.data_quality import ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(verbose=False)
spark = get_spark_session("05-data-viz-pix")

In [ ]:
monthly_df = spark.read.parquet(str(PIX_MONTHLY_INDICATORS_DIR))
card_savings_df = spark.read.parquet(str(PIX_FEE_SAVINGS_DIR))
transfer_savings_df = spark.read.parquet(str(PIX_TRANSFER_SAVINGS_DIR))

ensure_not_empty(monthly_df, "Gold indicadores mensais")
ensure_not_empty(card_savings_df, "Gold economia cartao")
ensure_not_empty(transfer_savings_df, "Gold economia transferencia")

monthly_pd = monthly_df.orderBy("ano_mes").toPandas()
card_pd = card_savings_df.orderBy("ano_mes", "cenario", "tipo_taxa").toPandas()
transfer_pd = transfer_savings_df.orderBy("ano_mes", "cenario", "tipo_taxa").toPandas()

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator

SOURCE_TEXT = "Fonte: dados públicos do Banco Central do Brasil"
SCENARIO_NOTE = "Estimativa hipotética baseada em cenários"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.labelsize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})


def format_billion(value, _position=None) -> str:
    return f"{value / 1_000_000_000:.1f}"


def format_currency(value, _position=None) -> str:
    return f"R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def select_xticks(labels, max_ticks: int = 8):
    labels = list(labels)
    if len(labels) <= max_ticks:
        return range(len(labels)), labels
    step = max(1, len(labels) // max_ticks)
    positions = list(range(0, len(labels), step))
    if positions[-1] != len(labels) - 1:
        positions.append(len(labels) - 1)
    return positions, [labels[index] for index in positions]


def add_max_annotation(ax, x_values, y_values, label_suffix: str = "") -> None:
    if len(y_values) == 0:
        return
    max_index = int(y_values.idxmax()) if hasattr(y_values, "idxmax") else max(range(len(y_values)), key=lambda i: y_values[i])
    x_value = x_values.iloc[max_index] if hasattr(x_values, "iloc") else x_values[max_index]
    y_value = y_values.iloc[max_index] if hasattr(y_values, "iloc") else y_values[max_index]
    ax.scatter([x_value], [y_value], color="#d62728", s=60, zorder=5)
    ax.annotate(
        f"Maior valor{label_suffix}",
        xy=(x_value, y_value),
        xytext=(10, 12),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": "#555555", "lw": 1},
        fontsize=10,
        color="#333333",
    )


def finish_plot(ax, title: str, xlabel: str, ylabel: str, output_name: str, legend: bool = False, note: str | None = None) -> None:
    ax.set_title(title, pad=16, weight="bold")
    ax.set_xlabel(xlabel, labelpad=10)
    ax.set_ylabel(ylabel, labelpad=10)
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.yaxis.offsetText.set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=7))
    ax.tick_params(axis="x", rotation=0)
    if legend:
        ax.legend(loc="upper left", frameon=False, ncol=1)
    footer = SOURCE_TEXT if note is None else f"{SOURCE_TEXT} | {note}"
    ax.figure.text(0.01, 0.015, footer, ha="left", va="bottom", fontsize=9, color="#555555")
    ax.figure.tight_layout(rect=(0, 0.06, 1, 1))
    output_path = FIGURES_DIR / output_name
    ax.figure.savefig(output_path, dpi=180)
    plt.close(ax.figure)
    print(f"Gráfico salvo em: {output_path.relative_to(PROJECT_DIR)}")

## Evolução Mensal da Quantidade de Transações Pix

In [ ]:
plot_df = monthly_pd.copy()
plot_df["quantidade_transacoes_bi"] = plot_df["quantidade_transacoes"] / 1_000_000_000
positions, labels = select_xticks(plot_df["ano_mes"])

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(plot_df["ano_mes"], plot_df["quantidade_transacoes_bi"], marker="o", linewidth=2.5, label="Transações Pix")
ax.set_xticks(positions)
ax.set_xticklabels(labels)
add_max_annotation(ax, plot_df["ano_mes"], plot_df["quantidade_transacoes_bi"], "")
finish_plot(
    ax,
    "Evolução mensal da quantidade de transações Pix",
    "Ano-mês",
    "Quantidade de transações (bilhões)",
    "01_pix_monthly_transactions.png",
    legend=True,
)

## Evolução Mensal do Volume Financeiro Pix

In [ ]:
plot_df = monthly_pd.copy()
plot_df["valor_total_bi"] = plot_df["valor_total"] / 1_000_000_000
positions, labels = select_xticks(plot_df["ano_mes"])

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(plot_df["ano_mes"], plot_df["valor_total_bi"], marker="o", linewidth=2.5, color="#1f77b4", label="Valor movimentado")
ax.set_xticks(positions)
ax.set_xticklabels(labels)
add_max_annotation(ax, plot_df["ano_mes"], plot_df["valor_total_bi"], "")
finish_plot(
    ax,
    "Evolução mensal do volume financeiro Pix",
    "Ano-mês",
    "Valor financeiro (R$ bilhões)",
    "02_pix_monthly_value.png",
    legend=True,
)

## Ticket Médio Mensal do Pix

In [ ]:
plot_df = monthly_pd.copy()
positions, labels = select_xticks(plot_df["ano_mes"])

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(plot_df["ano_mes"], plot_df["ticket_medio"], marker="o", linewidth=2.5, color="#2ca02c", label="Ticket médio")
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: format_currency(value)))
add_max_annotation(ax, plot_df["ano_mes"], plot_df["ticket_medio"], "")
finish_plot(
    ax,
    "Ticket médio mensal do Pix",
    "Ano-mês",
    "Ticket médio (R$)",
    "03_pix_average_ticket.png",
    legend=True,
)

## Economia Estimada com Taxas de Cartão

In [ ]:
card_plot = card_pd.groupby(["ano_mes", "cenario"], as_index=False)["economia_estimada"].sum()
card_plot["economia_estimada_bi"] = card_plot["economia_estimada"] / 1_000_000_000
positions, labels = select_xticks(sorted(card_plot["ano_mes"].unique()))

fig, ax = plt.subplots(figsize=(14, 7))
for cenario, group in card_plot.groupby("cenario"):
    ordered_group = group.sort_values("ano_mes")
    ax.plot(ordered_group["ano_mes"], ordered_group["economia_estimada_bi"], marker="o", linewidth=2.2, label=cenario)

max_row = card_plot.loc[card_plot["economia_estimada_bi"].idxmax()]
ax.scatter([max_row["ano_mes"]], [max_row["economia_estimada_bi"]], color="#d62728", s=60, zorder=5)
ax.annotate(
    "Maior valor",
    xy=(max_row["ano_mes"], max_row["economia_estimada_bi"]),
    xytext=(10, 12),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#555555", "lw": 1},
    fontsize=10,
    color="#333333",
)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
finish_plot(
    ax,
    "Economia potencial estimada com taxas de cartão",
    "Ano-mês",
    "Economia estimada (R$ bilhões)",
    "04_pix_estimated_card_fee_savings.png",
    legend=True,
    note=SCENARIO_NOTE,
)

## Economia Estimada com Transferências Tradicionais

In [ ]:
transfer_plot = transfer_pd.groupby(["ano_mes", "cenario"], as_index=False)["economia_estimada"].sum()
transfer_plot["economia_estimada_bi"] = transfer_plot["economia_estimada"] / 1_000_000_000
positions, labels = select_xticks(sorted(transfer_plot["ano_mes"].unique()))

fig, ax = plt.subplots(figsize=(14, 7))
for cenario, group in transfer_plot.groupby("cenario"):
    ordered_group = group.sort_values("ano_mes")
    ax.plot(ordered_group["ano_mes"], ordered_group["economia_estimada_bi"], marker="o", linewidth=2.2, label=cenario)

max_row = transfer_plot.loc[transfer_plot["economia_estimada_bi"].idxmax()]
ax.scatter([max_row["ano_mes"]], [max_row["economia_estimada_bi"]], color="#d62728", s=60, zorder=5)
ax.annotate(
    "Maior valor",
    xy=(max_row["ano_mes"], max_row["economia_estimada_bi"]),
    xytext=(10, 12),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#555555", "lw": 1},
    fontsize=10,
    color="#333333",
)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
finish_plot(
    ax,
    "Economia potencial estimada com transferências tradicionais",
    "Ano-mês",
    "Economia estimada (R$ bilhões)",
    "05_pix_estimated_transfer_fee_savings.png",
    legend=True,
    note=SCENARIO_NOTE,
)

In [ ]:
spark.stop()